[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/20_clustering/03_hierarchical_clustering.ipynb)

# Bài 03 — Hierarchical clustering

**🎯 Mục tiêu**
- **Agglomerative clustering**: bắt đầu mỗi điểm một cụm, gộp dần hai cụm gần nhất → cây phân cấp (**dendrogram**).
- **Divisive clustering** (chiều ngược lại, *top-down*): bắt đầu với một cụm chứa tất cả rồi tách dần — vd lặp lại K-Means với k = 2 trên cụm lớn nhất (*bisecting K-Means*, `sklearn.cluster.BisectingKMeans`). Ít phổ biến hơn vì mỗi bước tách tốn kém và quyết định tách sai ở đầu không sửa lại được; bài này tập trung vào agglomerative.
- Các kiểu **linkage**: single, complete, average, **Ward** — ảnh hưởng tới hình dạng cụm.
- Cài đặt from scratch (ma trận khoảng cách cụm) và đối chiếu `scipy.cluster.hierarchy`.
- Cắt dendrogram để chọn số cụm; ưu nhược điểm (không cần K trước, $O(n^2)$ bộ nhớ).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/20_clustering

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import AgglomerativeClustering
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score

from ml_course.visualization import PALETTE, set_style

set_style()
rng = np.random.default_rng(0)

## 1. Ý tưởng

1. Mỗi điểm là một cụm.
2. Lặp $n - 1$ lần: tìm **hai cụm gần nhất**, gộp lại.
3. Ghi lại thứ tự gộp và khoảng cách lúc gộp → dendrogram.

"Khoảng cách giữa hai cụm" (linkage) $A$, $B$:

| Linkage | $d(A, B)$ | Đặc điểm |
|---|---|---|
| Single | $\min_{a \in A, b \in B} d(a, b)$ | bắt hình dạng kéo dài; **chaining** — nối thành chuỗi qua nhiễu |
| Complete | $\max d(a, b)$ | cụm gọn, nhạy outlier |
| Average | trung bình mọi cặp | trung gian |
| **Ward** | mức tăng tổng bình phương trong cụm khi gộp | giống K-Means (cụm tròn, cân), thường mặc định tốt |

## 2. From scratch (single & complete linkage)

In [ ]:
def agglomerative(X, n_clusters, method="single"):
    clusters = {i: [i] for i in range(len(X))}
    D = np.sqrt(((X[:, None] - X[None]) ** 2).sum(-1))
    merges = []
    while len(clusters) > n_clusters:
        keys = list(clusters)
        best = (None, None, np.inf)
        for ii in range(len(keys)):
            for jj in range(ii + 1, len(keys)):
                block = D[np.ix_(clusters[keys[ii]], clusters[keys[jj]])]
                d = block.min() if method == "single" else block.max()
                if d < best[2]:
                    best = (keys[ii], keys[jj], d)
        a, b, d = best
        clusters[a] = clusters[a] + clusters.pop(b)
        merges.append(d)
    labels = np.empty(len(X), int)
    for c, members in enumerate(clusters.values()):
        labels[members] = c
    return labels, merges


Xs, ys = make_blobs(n_samples=90, centers=3, cluster_std=0.8, random_state=4)
for method in ["single", "complete"]:
    ours, _ = agglomerative(Xs, 3, method)
    ref = AgglomerativeClustering(3, linkage=method).fit_predict(Xs)
    print(f"{method:8s}: ARI(scratch, sklearn) = {adjusted_rand_score(ref, ours):.3f} | ARI so với nhãn thật = {adjusted_rand_score(ys, ours):.3f}")

Phiên bản ngây thơ trên tốn $O(n^3)$ thời gian — chỉ để hiểu thuật toán. scipy/sklearn dùng thuật toán tối ưu (cập nhật Lance-Williams, nearest-neighbor chain) với $O(n^2)$ — nhưng vẫn cần **ma trận khoảng cách $n \times n$** → khó dùng khi $n$ > vài chục nghìn.

## 3. Dendrogram và cắt cây

In [ ]:
Z = linkage(Xs, method="ward")
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
dendrogram(Z, ax=axes[0], color_threshold=10, no_labels=True)
axes[0].axhline(10, color="r", ls="--", label="cắt ở độ cao 10 → 3 cụm")
axes[0].set_ylabel("khoảng cách khi gộp (Ward)")
axes[0].legend()
axes[0].set_title("Dendrogram")
lab = fcluster(Z, t=10, criterion="distance")
axes[1].scatter(*Xs.T, c=[PALETTE[l - 1] for l in lab], s=25)
axes[1].set_title(f"Cắt cây: {lab.max()} cụm, ARI = {adjusted_rand_score(ys, lab):.2f}")
plt.show()

Đọc dendrogram: chiều cao mỗi điểm gộp = "giá" của việc gộp. **Khoảng trống dọc lớn** (gộp tiếp tốn rất nhiều) gợi ý số cụm tự nhiên. Một lần chạy cho **mọi** số cụm — chỉ cần cắt ở độ cao khác nhau.

## 4. Linkage thay đổi kết quả

In [ ]:
datasets = {"mặt trăng": make_moons(300, noise=0.05, random_state=0),
            "blobs + nhiễu nối": (np.vstack([make_blobs(300, centers=[[0, 0], [5, 0]], cluster_std=0.6, random_state=1)[0],
                                             np.c_[np.linspace(0.8, 4.2, 20), rng.normal(0, 0.05, 20)]]),
                                  np.r_[make_blobs(300, centers=[[0, 0], [5, 0]], cluster_std=0.6, random_state=1)[1], np.full(20, -1)])}
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
for row, (dname, (Xd, yd)) in zip(axes, datasets.items()):
    for ax, method in zip(row, ["single", "complete", "average", "ward"]):
        lab = AgglomerativeClustering(2, linkage=method).fit_predict(Xd)
        mask = yd >= 0
        ax.scatter(*Xd.T, c=[PALETTE[l] for l in lab], s=10)
        ax.set_title(f"{dname} — {method}: ARI = {adjusted_rand_score(yd[mask], lab[mask]):.2f}", fontsize=10)
plt.tight_layout()
plt.show()

- **Single linkage** tách hoàn hảo hai mặt trăng (nối theo "chuỗi" láng giềng gần) — nhưng cũng chính vì thế mà một "cây cầu" nhiễu mảnh giữa hai cụm blobs làm nó gộp tất cả thành một (**chaining**).
- **Ward/complete** cho cụm gọn: tốt với blobs, thất bại với mặt trăng.
- Không có linkage "tốt nhất" — phụ thuộc hình dạng cụm ta mong đợi.

## 5. Khi nào dùng hierarchical clustering?

- Muốn **cấu trúc phân cấp** (phân loại sinh học, taxonomy sản phẩm, nhóm tài liệu).
- Không biết trước K — xem dendrogram rồi quyết định.
- Dữ liệu nhỏ–vừa (≤ ~10–20 nghìn điểm), hoặc chỉ có **ma trận khoảng cách** (khoảng cách tuỳ ý: edit distance, DTW...).
- Cần ràng buộc kết nối (`connectivity=`) — vd phân vùng ảnh/bản đồ chỉ gộp vùng kề nhau.

## 🧠 Kiểm tra nhanh

1. Single linkage và complete linkage khác nhau thế nào? Hiện tượng chaining là gì?
2. Vì sao hierarchical clustering khó áp dụng cho 1 triệu điểm?
3. Đọc dendrogram thế nào để chọn số cụm?
4. Ward linkage giống thuật toán nào đã học?

## 📌 Tóm tắt

- Agglomerative: gộp dần hai cụm gần nhất; dendrogram cho mọi số cụm.
- Divisive (top-down, vd bisecting K-Means) là chiều ngược lại: tách dần từ một cụm.
- Linkage quyết định hình dạng cụm: single (chuỗi), complete/Ward (gọn), average (trung gian).
- $O(n^2)$ bộ nhớ; hợp dữ liệu nhỏ–vừa, cần phân cấp hoặc khoảng cách tuỳ ý.

## 📚 Tài liệu tham khảo

- Müllner (2011) — *Modern hierarchical, agglomerative clustering algorithms*
- ESL §14.3.12
- scipy.cluster.hierarchy documentation